# TF-IDF + Multinomial Naive Bayes: BABE-only vs BABE + Wikipedia augmentation

This notebook compares two training conditions using the **same saved BABE splits**:

- **Condition A:** TF-IDF + Multinomial Naive Bayes trained on BABE training data only.
- **Condition B:** TF-IDF + Multinomial Naive Bayes trained on BABE training data plus a newly sampled, balanced Wikipedia-derived subset from MBIB.

**Important:** The BABE + Wikipedia examples are **not assumed to be saved already**. The notebook downloads the MBIB `linguistic_bias` split, filters the Wikipedia-derived sources (dataset IDs `3` and `26`), removes text overlap with all BABE splits, and samples up to 2,000 examples per class (4,000 total expected). It optionally saves the generated sample for reproducibility, but does not require a pre-existing sample file.

Both conditions use the same BABE validation and test splits. `alpha` is selected using validation macro-F1, and the test set is evaluated only after tuning. Do not tune choices based on test performance.

**Comparability note:** this uses the saved `babe_sg2_splits` split files. Confirm exact test-row identity before comparing scores with results produced using a different split.

## 1. Install dependencies

In [ ]:
%pip install -q datasets scikit-learn pandas==2.2.3

## 2. Mount Drive and set configuration

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json, random
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)

SEED = 42
LABEL2ID = {"Non-biased": 0, "Biased": 1}
ID2LABEL = {0: "Non-biased", 1: "Biased"}

SPLIT_DIR = Path("/content/drive/MyDrive/NewsBias_Project_Backup/babe_sg2_splits")
OUTPUT_DIR = Path("/content/drive/MyDrive/NewsBias_Project_Backup/tfidf_naive_bayes_experiment")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# TF-IDF settings held constant across conditions.
MAX_FEATURES = 100_000
NGRAM_RANGE = (1, 2)
MIN_DF = 2
MAX_DF = 0.98
SUBLINEAR_TF = True
ALPHAS = [0.01, 0.05, 0.1, 0.5, 1.0, 2.0]

random.seed(SEED)
np.random.seed(SEED)
print("Output folder:", OUTPUT_DIR)

Mounted at /content/drive
Output folder: /content/drive/MyDrive/NewsBias_Project_Backup/tfidf_naive_bayes_experiment


## 3. Load the existing BABE splits — do not regenerate them

In [ ]:
assert all((SPLIT_DIR / f"{name}.csv").exists()
           for name in ["train", "validation", "test"]), (
    f"Expected train.csv, validation.csv, and test.csv inside {SPLIT_DIR}. "
    "Copy the saved babe_sg2_splits folder to this Drive location first."
)

train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df = pd.read_csv(SPLIT_DIR / "validation.csv")
test_df = pd.read_csv(SPLIT_DIR / "test.csv")

def normalize_binary_label(v):
    if pd.isna(v):
        return np.nan
    if isinstance(v, (int, np.integer, float, np.floating)) and v in (0, 1):
        return int(v)
    s = str(v).strip().lower().replace("_", "-")
    mapping = {
        "0": 0, "non-biased": 0, "non biased": 0,
        "nonbiased": 0, "unbiased": 0, "1": 1, "biased": 1
    }
    return mapping.get(s, np.nan)

for name, frame in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    if "label" not in frame.columns and "label_bias" in frame.columns:
        frame["label"] = frame["label_bias"].map(normalize_binary_label)
    else:
        frame["label"] = frame["label"].map(normalize_binary_label)
    assert frame["label"].notna().all(), f"Missing or unexpected labels in {name}"
    frame["label"] = frame["label"].astype(int)
    frame["text"] = frame["text"].fillna("").astype(str).str.strip()
    assert frame["text"].str.len().gt(0).all(), f"Empty text in {name}"

for name, frame in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"{name}: {len(frame)} rows | label counts: {frame['label'].value_counts().sort_index().to_dict()}")

Train: 2579 rows | label counts: {0: 1327, 1: 1252}
Validation: 550 rows | label counts: {0: 266, 1: 284}
Test: 508 rows | label counts: {0: 252, 1: 256}


## 4. Build the Wikipedia-derived augmentation sample from MBIB

In [ ]:
# Download/build the augmentation set during this notebook run.
# No pre-saved BABE+Wikipedia file is required.
mbib_ds = load_dataset("mediabiasgroup/mbib-base", split="linguistic_bias")
mbib = mbib_ds.to_pandas()

print("MBIB rows:", len(mbib))
print("Columns:", mbib.columns.tolist())
assert {"text", "label", "dataset_id"}.issubset(mbib.columns), (
    "MBIB schema differs from expected; inspect the dataset card and update this cell."
)
display(pd.crosstab(mbib["dataset_id"], mbib["label"], margins=True))

# Source IDs 3 = WikiNPOV and 26 = Wiki Neutrality Corpus.
wiki = mbib[mbib["dataset_id"].astype(str).isin(["3", "26"])].copy()
wiki["text"] = wiki["text"].fillna("").astype(str).str.strip()
wiki["label"] = pd.to_numeric(wiki["label"], errors="coerce")
wiki = wiki.dropna(subset=["text", "label"])
wiki = wiki[wiki["label"].isin([0, 1])].copy()
wiki["label"] = wiki["label"].astype(int)
wiki = wiki[wiki["text"].str.len().between(20, 1500)].copy()

# Normalize for deduplication and remove exact text overlap against all BABE splits.
wiki["text_norm"] = wiki["text"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
wiki = wiki.drop_duplicates("text_norm")
babe_texts = set(
    pd.concat([train_df["text"], val_df["text"], test_df["text"]])
    .astype(str).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
)
wiki = wiki[~wiki["text_norm"].isin(babe_texts)].copy()

MAX_PER_LABEL = 2000
sample_parts = []
for label in [0, 1]:
    part = wiki[wiki["label"] == label]
    n = min(MAX_PER_LABEL, len(part))
    sample_parts.append(part.sample(n=n, random_state=SEED + label))

wiki_sample = (
    pd.concat(sample_parts, ignore_index=True)
      .sample(frac=1, random_state=SEED)
      .reset_index(drop=True)
)
wiki_sample = wiki_sample.drop(columns=["text_norm"], errors="ignore")

print("Generated Wikipedia sample:", len(wiki_sample))
print("Label counts:", wiki_sample["label"].value_counts().sort_index().to_dict())
display(wiki_sample.groupby(["dataset_id", "label"]).size().rename("n").reset_index())

if len(wiki_sample) < 4000:
    print("WARNING: fewer than 4,000 eligible examples were available. The notebook will continue with the sample obtained.")
else:
    assert len(wiki_sample) == 4000

# Save the generated sample as an artifact for reproducibility; this file is CREATED here.
wiki_sample.to_csv(OUTPUT_DIR / "wiki_augmentation_sample_generated.csv", index=False)
print("Generated sample saved to:", OUTPUT_DIR / "wiki_augmentation_sample_generated.csv")

README.md:   0%|          | 0.00/4.22k [00:00<?, ?B/s]

cognitive-bias.csv:   0%|          | 0.00/851k [00:00<?, ?B/s]

fake-news.csv:   0%|          | 0.00/1.00M [00:00<?, ?B/s]

gender-bias.csv:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

mbib-aggregated/hate-speech.csv: reconstructing file:   0%|          |  0.00B / 99.0MB            

...

MBIB rows: 401862
Columns: ['id', 'text', 'label', 'dataset_id']


label,0,1,All
dataset_id,,,
3,10068,1662,11730
9,0,1444,1444
10,1862,1618,3480
26,178228,160833,339061
35,5842,28731,34573
38,37,683,720
75,4894,5960,10854
All,200931,200931,401862


Generated Wikipedia sample: 4000
Label counts: {0: 2000, 1: 2000}


,dataset_id,label,n
0,3,0,103
1,3,1,18
2,26,0,1897
3,26,1,1982


Generated sample saved to: /content/drive/MyDrive/NewsBias_Project_Backup/tfidf_naive_bayes_experiment/wiki_augmentation_sample_generated.csv


## 5. Define TF-IDF + Naive Bayes evaluation

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

def make_vectorizer():
    return TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=1,
        max_df=1.0,
        token_pattern=r"(?u)\b\w+\b",
        sublinear_tf=True,
        max_features=100_000
    )

def tune_and_evaluate(condition, train_frame):
    # Fit vocabulary/IDF only on this condition's training text.
    vectorizer = make_vectorizer()
    X_train = vectorizer.fit_transform(train_frame["text"].astype(str))
    y_train = train_frame["label"].astype(int).to_numpy()
    X_val = vectorizer.transform(val_df["text"].astype(str))
    y_val = val_df["label"].astype(int).to_numpy()
    X_test = vectorizer.transform(test_df["text"].astype(str))
    y_test = test_df["label"].astype(int).to_numpy()

    # Select smoothing alpha using validation macro-F1, never test performance.
    tuning_rows = []
    best_alpha, best_val_f1 = None, -1.0
    for alpha in ALPHAS:
        model = MultinomialNB(alpha=alpha)
        model.fit(X_train, y_train)
        val_pred = model.predict(X_val)
        val_f1 = f1_score(y_val, val_pred, average="macro", zero_division=0)
        val_acc = accuracy_score(y_val, val_pred)
        tuning_rows.append({
            "condition": condition, "alpha": alpha,
            "validation_macro_f1": val_f1, "validation_accuracy": val_acc
        })
        if val_f1 > best_val_f1:
            best_val_f1, best_alpha = val_f1, alpha

    tuning_df = pd.DataFrame(tuning_rows)
    display(tuning_df.sort_values("validation_macro_f1", ascending=False))
    best_model = MultinomialNB(alpha=best_alpha)
    best_model.fit(X_train, y_train)
    y_pred = best_model.predict(X_test)

    print(f"\n--- {condition} ---")
    print("Training rows:", len(train_frame))
    print("Vocabulary size:", len(vectorizer.vocabulary_))
    print("Selected alpha:", best_alpha)
    print("Best validation macro-F1:", round(best_val_f1, 4))
    print("\nTest classification report:")
    print(classification_report(
        y_test, y_pred, labels=[0, 1],
        target_names=["Non-biased", "Biased"], zero_division=0
    ))
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    print("Confusion matrix (rows=true [Non-biased, Biased], columns=predicted [Non-biased, Biased]):")
    print(cm)

    metrics = {
        "condition": condition,
        "model": "TF-IDF + MultinomialNB",
        "train_size": int(len(train_frame)),
        "vocabulary_size": int(len(vectorizer.vocabulary_)),
        "selected_alpha": float(best_alpha),
        "best_validation_macro_f1": float(best_val_f1),
        "test_size": int(len(y_test)),
        "test_accuracy": float(accuracy_score(y_test, y_pred)),
        "test_macro_f1": float(f1_score(y_test, y_pred, average="macro", zero_division=0)),
        "test_nonbiased_f1": float(f1_score(y_test, y_pred, pos_label=0, zero_division=0)),
        "test_biased_f1": float(f1_score(y_test, y_pred, pos_label=1, zero_division=0)),
        "test_biased_precision": float(precision_score(y_test, y_pred, pos_label=1, zero_division=0)),
        "test_biased_recall": float(recall_score(y_test, y_pred, pos_label=1, zero_division=0)),
        "tn": int(cm[0, 0]), "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]), "tp": int(cm[1, 1])
    }

    pred_df = test_df[["text", "label"]].copy()
    pred_df = pred_df.rename(columns={"label": "true_label"})
    pred_df["predicted_label"] = y_pred
    pred_df["true_label_name"] = pred_df["true_label"].map(ID2LABEL)
    pred_df["predicted_label_name"] = pred_df["predicted_label"].map(ID2LABEL)

    # Persist condition artifacts.
    safe_name = condition.lower().replace(" ", "_").replace("+", "plus")
    pred_df.to_csv(OUTPUT_DIR / f"{safe_name}_test_predictions.csv", index=False)
    tuning_df.to_csv(OUTPUT_DIR / f"{safe_name}_validation_tuning.csv", index=False)
    with open(OUTPUT_DIR / f"{safe_name}_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    train_frame = train_frame.copy()

    train_frame["text"] = (
        train_frame["text"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    train_frame = train_frame[train_frame["text"] != ""].copy()

    if train_frame.empty:
        raise ValueError(
            f"{condition}: No non-empty training texts found. "
            "Check dataset loading and preprocessing."
        )

    print(f"{condition}: {len(train_frame)} training rows")
    print(train_frame["text"].head(3).to_list())

    return metrics, pred_df, tuning_df

In [ ]:
print("Training rows:", len(train_df))
print("Columns:", train_df.columns.tolist())
print("\nFirst 5 texts:")
print(train_df["text"].head().to_list())

print("\nMissing texts:", train_df["text"].isna().sum())
print("Empty texts:", train_df["text"].fillna("").astype(str).str.strip().eq("").sum())

print("\nText lengths:")
print(train_df["text"].fillna("").astype(str).str.len().describe())

print("\nSample labels:")
print(train_df["label"].value_counts(dropna=False))

Training rows: 2579
Columns: ['text', 'news_link', 'outlet', 'topic', 'type', 'label_bias', 'label_opinion', 'biased_words', 'label']

First 5 texts:
['"Orange Is the New Black" star Yael Stone is renouncing her U.S. green card to return to her native Australia in order to fight climate change.', '...immigrants as criminals and eugenics, all of which were once considered fringe and extreme. White nationalists embrace white supremacist and white separatist views.', '...we sounded the alarm in the early months of Trump’s presidency, when prime-time Fox News host Tucker Carlson began to experiment with segments that used euphemisms like “Western civilization” to package the idea that white people are inherently more civilized while people of color are a threat to national stability.', '[Black Lives Matter] is essentially a non-falsifiable religious cult that sets itself specifically against Christianity, just like the French Revolution did.', '[The police] now prefer to think of themselve

## 6. Condition A — train on BABE only

In [ ]:
result_babe_only, preds_babe_only, tuning_babe_only = tune_and_evaluate(
    "BABE-only", train_df[["text", "label"]].copy()
)

,condition,alpha,validation_macro_f1,validation_accuracy
2,BABE-only,0.10,0.719629,0.720000
1,BABE-only,0.05,0.716060,0.716364
3,BABE-only,0.50,0.708199,0.710909
0,BABE-only,0.01,0.699880,0.700000
4,BABE-only,1.00,0.688099,0.692727
5,BABE-only,2.00,0.686520,0.694545



--- BABE-only ---
Training rows: 2579
Vocabulary size: 61402
Selected alpha: 0.1
Best validation macro-F1: 0.7196

Test classification report:
              precision    recall  f1-score   support

  Non-biased       0.66      0.79      0.71       252
      Biased       0.74      0.59      0.66       256

    accuracy                           0.69       508
   macro avg       0.70      0.69      0.69       508
weighted avg       0.70      0.69      0.69       508

Confusion matrix (rows=true [Non-biased, Biased], columns=predicted [Non-biased, Biased]):
[[198  54]
 [104 152]]
BABE-only: 2579 training rows
['"Orange Is the New Black" star Yael Stone is renouncing her U.S. green card to return to her native Australia in order to fight climate change.', '...immigrants as criminals and eugenics, all of which were once considered fringe and extreme. White nationalists embrace white supremacist and white separatist views.', '...we sounded the alarm in the early months of Trump’s presidency

## 7. Condition B — train on BABE + generated Wikipedia sample

In [ ]:
train_aug_frame = pd.concat(
    [train_df[["text", "label"]], wiki_sample[["text", "label"]]],
    ignore_index=True
)
print("BABE training rows:", len(train_df))
print("Wikipedia augmentation rows:", len(wiki_sample))
print("Combined training rows:", len(train_aug_frame))
print("Combined label counts:", train_aug_frame["label"].value_counts().sort_index().to_dict())

result_augmented, preds_augmented, tuning_augmented = tune_and_evaluate(
    "BABE+Wikipedia", train_aug_frame
)

BABE training rows: 2579
Wikipedia augmentation rows: 4000
Combined training rows: 6579
Combined label counts: {0: 3327, 1: 3252}


,condition,alpha,validation_macro_f1,validation_accuracy
2,BABE+Wikipedia,0.10,0.714204,0.714545
5,BABE+Wikipedia,2.00,0.703557,0.705455
1,BABE+Wikipedia,0.05,0.703471,0.703636
4,BABE+Wikipedia,1.00,0.700233,0.701818
3,BABE+Wikipedia,0.50,0.695396,0.696364
0,BABE+Wikipedia,0.01,0.683636,0.683636



--- BABE+Wikipedia ---
Training rows: 6579
Vocabulary size: 100000
Selected alpha: 0.1
Best validation macro-F1: 0.7142

Test classification report:
              precision    recall  f1-score   support

  Non-biased       0.67      0.79      0.73       252
      Biased       0.75      0.62      0.68       256

    accuracy                           0.70       508
   macro avg       0.71      0.71      0.70       508
weighted avg       0.71      0.70      0.70       508

Confusion matrix (rows=true [Non-biased, Biased], columns=predicted [Non-biased, Biased]):
[[198  54]
 [ 96 160]]
BABE+Wikipedia: 6579 training rows
['"Orange Is the New Black" star Yael Stone is renouncing her U.S. green card to return to her native Australia in order to fight climate change.', '...immigrants as criminals and eugenics, all of which were once considered fringe and extreme. White nationalists embrace white supremacist and white separatist views.', '...we sounded the alarm in the early months of Trump’s

## 8. Compare conditions and save a combined summary

In [ ]:
comparison = pd.DataFrame([result_babe_only, result_augmented])
display(comparison)

for metric in ["test_accuracy", "test_macro_f1", "test_biased_f1", "test_biased_precision", "test_biased_recall"]:
    delta = result_augmented[metric] - result_babe_only[metric]
    print(f"BABE+Wikipedia minus BABE-only {metric}: {delta:+.4f} ({delta*100:+.2f} percentage points)")

comparison.to_csv(OUTPUT_DIR / "tfidf_nb_condition_comparison.csv", index=False)
with open(OUTPUT_DIR / "experiment_config.json", "w") as f:
    json.dump({
        "seed": SEED,
        "model": "TfidfVectorizer + MultinomialNB",
        "tfidf": {
            "max_features": MAX_FEATURES,
            "ngram_range": list(NGRAM_RANGE),
            "min_df": MIN_DF,
            "max_df": MAX_DF,
            "sublinear_tf": SUBLINEAR_TF
        },
        "alpha_candidates": ALPHAS,
        "train_rows_babe_only": int(len(train_df)),
        "train_rows_augmented": int(len(train_aug_frame)),
        "validation_rows": int(len(val_df)),
        "test_rows": int(len(test_df)),
        "augmentation_rows": int(len(wiki_sample)),
        "augmentation_source_ids": sorted(wiki_sample["dataset_id"].astype(str).unique().tolist()),
        "note": "Wikipedia labels are distant supervision. Augmentation sample is generated during notebook execution; results should be interpreted empirically."
    }, f, indent=2)
print("All outputs saved under:", OUTPUT_DIR)

,condition,model,train_size,vocabulary_size,selected_alpha,best_validation_macro_f1,test_size,test_accuracy,test_macro_f1,test_nonbiased_f1,test_biased_f1,test_biased_precision,test_biased_recall,tn,fp,fn,tp
0,BABE-only,TF-IDF + MultinomialNB,2579,61402,0.1,0.719629,508,0.688976,0.686405,0.714801,0.658009,0.737864,0.59375,198,54,104,152
1,BABE+Wikipedia,TF-IDF + MultinomialNB,6579,100000,0.1,0.714204,508,0.704724,0.703063,0.725275,0.680851,0.747664,0.62500,198,54,96,160


BABE+Wikipedia minus BABE-only test_accuracy: +0.0157 (+1.57 percentage points)
BABE+Wikipedia minus BABE-only test_macro_f1: +0.0167 (+1.67 percentage points)
BABE+Wikipedia minus BABE-only test_biased_f1: +0.0228 (+2.28 percentage points)
BABE+Wikipedia minus BABE-only test_biased_precision: +0.0098 (+0.98 percentage points)
BABE+Wikipedia minus BABE-only test_biased_recall: +0.0312 (+3.12 percentage points)
All outputs saved under: /content/drive/MyDrive/NewsBias_Project_Backup/tfidf_naive_bayes_experiment


## 9. Interpretation checklist

- Compare **macro-F1**, accuracy, class-specific F1, precision/recall, and the confusion matrix—not accuracy alone.
- A higher score on BABE+Wikipedia is not guaranteed. Wikipedia-derived labels are noisy/distant supervision and may not match BABE's annotation process.
- The validation set selects `alpha`; the test set is reserved for final evaluation.
- Because TF-IDF is fit separately for each condition, the augmented condition has its own vocabulary and IDF weights. This is expected.
- This is a baseline experiment. If you compare it with DeBERTa or other models, use the same saved BABE test split and report all metrics consistently.
- The output folder includes the generated augmentation sample, per-condition metrics, validation tuning tables, test predictions, and comparison CSV.